In [2]:
import pandas as pd

In [3]:
import pandas as pd

df = pd.read_csv("dataset\\data raw\\2026-installation-production-stockage-electricite-agrege.csv", sep=";", encoding="utf-8", low_memory=False,
                 dtype={"codeDepartement": str, "codeINSEECommune": str, "codeIRIS": str})

# 1. Quelles valeurs pour la filière et la technologie ?
print(df["filiere"].value_counts(dropna=False))
print(df[df["filiere"].str.contains("olaire", case=False, na=False)]
        ["technologie"].value_counts(dropna=False))

# 2. Filtre sur tes départements
deps = ["26", "07", "38", "05", "04", "84"]
sol = df[df["codeDepartement"].isin(deps) &
         df["filiere"].str.contains("olaire", case=False, na=False)].copy()
print(sol.shape)
print(sol.groupby("codeDepartement").size())

# 3. Structure des lignes et puissance
print(sol["nbInstallations"].value_counts().head(10))
print(sol["puisMaxInstallee"].describe())

# 4. Taux de remplissage des colonnes clés
cols = ["dateRaccordement", "dateMiseEnService", "dateDeraccordement",
        "energieAnnuelleGlissanteProduite", "energieAnnuelleGlissanteInjectee"]
print(sol[cols].notna().mean().round(3))

# 5. Doublons éventuels (versions multiples d'une même installation)
print(sol["codeEICResourceObject"].duplicated().sum())
print(sol["gestionnaire"].value_counts())

filiere
Solaire                       132856
Hydraulique                     2752
Eolien                          2458
Bioénergies                     1259
Thermique non renouvelable      1238
Stockage non hydraulique         922
Nucléaire                         57
Autre                             41
Energies Marines                   4
Géothermie                         2
Name: count, dtype: int64
technologie
Photovoltaïque     128273
NaN                  3372
Autre                1205
Thermodynamique         5
Batterie                1
Name: count, dtype: int64
(8736, 50)
codeDepartement
04    1023
05    1004
07    1066
26    2143
38    2128
84    1372
dtype: int64
nbInstallations
1     6332
11     106
10     101
13     100
14      96
18      88
12      83
16      77
19      71
20      67
Name: count, dtype: int64
count     8736.000000
mean       290.038972
std        901.016248
min         27.100000
25%         90.000000
50%        113.070000
75%        220.000000
max      16650.0

In [4]:
sol = df[df["codeDepartement"].isin(deps) & (df["filiere"] == "Solaire")].copy()
sol["agrege"] = sol["nbInstallations"] > 1

# A. Les lignes agrégées sont-elles celles sans EIC ni gestionnaire ?
print(pd.crosstab(sol["agrege"], sol["codeEICResourceObject"].isna()))
print(pd.crosstab(sol["agrege"], sol["gestionnaire"].isna()))

# B. Technologie dans ton périmètre
print(pd.crosstab(sol["technologie"].fillna("NaN"), sol["agrege"]))

# C. Seuil 36 kW sur les lignes individuelles
ind = sol[~sol["agrege"]].copy()
print((ind["puisMaxInstallee"] < 36).sum(), (ind["puisMaxRac"] < 36).sum())
print(ind[["puisMaxInstallee", "puisMaxRac"]].describe())

# D. Dates
cols_date = [c for c in sol.columns if c.lower().startswith("datemiseenservice")] + ["dateRaccordement"]
print(sol[cols_date].notna().mean())
for c in ["dateRaccordement", "dateMiseEnService"]:
    sol[c] = pd.to_datetime(sol[c], errors="coerce")
ind = sol[~sol["agrege"]].copy()
print(sol.groupby("agrege")["dateRaccordement"].describe())       # dates des lignes agrégées crédibles ?
print(ind["dateRaccordement"].dt.year.value_counts().sort_index())

# E. Unité de la production injectée
ind["kwh_par_kw"] = ind["energieAnnuelleGlissanteInjectee"] / ind["puisMaxInstallee"]
print(ind["kwh_par_kw"].describe())
print(ind.groupby("codeDepartement")[["puisMaxInstallee", "energieAnnuelleGlissanteInjectee"]].sum())
print(sol.groupby("agrege")["puisMaxInstallee"].sum())            # poids des petites installations

codeEICResourceObject  False  True 
agrege                             
False                   6332      0
True                       0   2404
gestionnaire  False  True 
agrege                    
False          6332      0
True              0   2404
agrege           False  True 
technologie                  
NaN                  9      2
Photovoltaïque    6322   2402
Thermodynamique      1      0
0 0
       puisMaxInstallee    puisMaxRac
count       6332.000000   6193.000000
mean         315.797310    314.146442
std         1038.128751   1048.661239
min           36.960000     37.000000
25%           90.000000     90.000000
50%          108.000000    100.000000
75%          220.000000    220.000000
max        16650.000000  16000.000000
dateMiseEnService                  0.999657
dateMiseEnservice (format date)    0.999657
dateRaccordement                   1.000000
dtype: float64
       count                           mean                  min  \
agrege                               

C:\Users\willi\AppData\Local\Temp\ipykernel_18608\1678582292.py:20: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  sol[c] = pd.to_datetime(sol[c], errors="coerce")
C:\Users\willi\AppData\Local\Temp\ipykernel_18608\1678582292.py:20: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  sol[c] = pd.to_datetime(sol[c], errors="coerce")


In [5]:
fmt = "%d/%m/%Y"   # format explicite : supprime l'avertissement et évite les erreurs jour/mois
for c in ["dateRaccordement", "dateMiseEnService"]:
    sol[c] = pd.to_datetime(sol[c], format=fmt, errors="coerce")

ind = sol[(~sol["agrege"]) & (sol["technologie"] == "Photovoltaïque")].copy()

# 1. Dates : conversions ratées ? écart entre raccordement et mise en service ?
print(ind[["dateRaccordement", "dateMiseEnService"]].isna().sum())
print((ind["dateMiseEnService"] - ind["dateRaccordement"]).dt.days.describe())

# 2. Effet des installations récentes sur la production
ind["annee_racc"] = ind["dateRaccordement"].dt.year
ind["kwh_par_kw"] = ind["energieAnnuelleGlissanteInjectee"] / ind["puisMaxInstallee"]
print(ind.groupby("annee_racc")["kwh_par_kw"].agg(["count", "median"]).tail(8))
print(ind["energieAnnuelleGlissanteInjectee"].isna().sum(),
      (ind["energieAnnuelleGlissanteInjectee"] == 0).sum())

# 3. Table centrale du projet : puissance cumulée (MW) par département et par année
annuel = (ind.groupby(["codeDepartement", "annee_racc"])["puisMaxInstallee"].sum()
             .unstack(fill_value=0)
             .reindex(columns=range(2007, 2027), fill_value=0)
             .cumsum(axis=1) / 1000)
print(annuel.round(1))

dateRaccordement     0
dateMiseEnService    3
dtype: int64
count    6319.000000
mean        0.411458
std        13.146436
min       -13.000000
25%         0.000000
50%         0.000000
75%         0.000000
max       480.000000
dtype: float64
            count       median
annee_racc                    
2019          224  1331.116667
2020          288  1293.061616
2021          437  1305.739583
2022          471  1316.867470
2023          546  1244.038148
2024          847  1283.185525
2025         1095  1278.705882
2026          640   603.945789
173 29
annee_racc       2007  2008  2009  2010   2011   2012   2013   2014   2015  \
codeDepartement                                                              
04                0.0   0.0   8.9  38.4  155.5  175.0  177.3  187.4  207.0   
05                0.0   0.0   0.7   2.0    7.8   10.4   12.9   23.6   31.4   
07                0.0   0.0   0.2   3.0   12.7   20.4   25.1   44.4   45.6   
26                0.0   0.0   0.7  18.7   41.1   52

In [1]:
import pandas as pd
from pathlib import Path

RAW = Path("dataset/data raw")   # adapte si ton dossier s'appelle autrement
DEPS = ["26", "07", "38", "05", "04", "84"]

def lire(path):
    """Teste séparateur et encodage jusqu'à obtenir un tableau cohérent."""
    for sep in [";", ","]:
        for enc in ["utf-8", "latin-1"]:
            try:
                df = pd.read_csv(path, sep=sep, encoding=enc, dtype=str, low_memory=False)
                if df.shape[1] > 5:
                    return df, sep, enc
            except Exception:
                continue
    raise ValueError(f"Lecture impossible : {path}")

infos, schemas, solaire = [], {}, {}
for p in sorted(RAW.glob("*.csv")):
    annee = int(p.name[:4])
    df, sep, enc = lire(p)
    schemas[annee] = set(df.columns)
    infos.append({"annee": annee, "fichier": p.name, "sep": sep, "enc": enc,
                  "lignes": len(df), "colonnes": df.shape[1]})
    # on ne garde en mémoire que le solaire de tes 6 départements, si les colonnes existent
    if {"codeDepartement", "filiere"} <= set(df.columns):
        solaire[annee] = df[df["codeDepartement"].isin(DEPS) & (df["filiere"] == "Solaire")].copy()
    else:
        print(f"{annee} : colonnes codeDepartement/filiere absentes")
    del df

pd.set_option("display.max_colwidth", None)
print(pd.DataFrame(infos).to_string(index=False))

ref = schemas[2026]
for a in sorted(schemas):
    print(f"\n{a} - manquantes vs 2026 :", sorted(ref - schemas[a]))
    print(f"{a} - en plus :", sorted(schemas[a] - ref))

 annee                                                      fichier sep   enc  lignes  colonnes
  2017 2017-installation-production-stockage-electricite-agrege.csv   ; utf-8   41237        47
  2018 2018-installation-production-stockage-electricite-agrege.csv   ; utf-8   44598        49
  2019 2019-installation-production-stockage-electricite-agrege.csv   ; utf-8   49043        47
  2020 2020-installation-production-stockage-electricite-agrege.csv   ; utf-8   54671        47
  2021 2021-installation-production-stockage-electricite-agrege.csv   ; utf-8   63410        49
  2022 2022-installation-production-stockage-electricite-agrege.csv   ; utf-8   72790        49
  2023 2023-installation-production-stockage-electricite-agrege.csv   ; utf-8   84944        50
  2024 2024-installation-production-stockage-electricite-agrege.csv   ; utf-8  101097        52
  2025 2025-installation-production-stockage-electricite-agrege.csv   ; utf-8  125930        51
  2026 2026-installation-production-stoc

In [6]:
def resume(s, annee):
    s = s.copy()
    for c in ["puisMaxInstallee", "energieAnnuelleGlissanteInjectee", "nbInstallations"]:
        if c in s:
            s[c] = pd.to_numeric(s[c].str.replace(",", ".", regex=False), errors="coerce")
    indiv = s[s["nbInstallations"] == 1] if "nbInstallations" in s else s
    return {
        "annee": annee,
        "lignes_solaire_6dep": len(s),
        "lignes_individuelles": len(indiv),
        "puissance_MW_indiv": round(indiv["puisMaxInstallee"].sum() / 1000, 1),
        "puissance_MW_agreg": round((s["puisMaxInstallee"].sum() - indiv["puisMaxInstallee"].sum()) / 1000, 1),
        "remplissage_dateRacc": round(s["dateRaccordement"].notna().mean(), 3) if "dateRaccordement" in s else None,
        "remplissage_injectee": round(s["energieAnnuelleGlissanteInjectee"].notna().mean(), 3) if "energieAnnuelleGlissanteInjectee" in s else None,
        "gestionnaires": sorted(indiv["gestionnaire"].dropna().unique()) if "gestionnaire" in indiv else None,
    }

tableau = pd.DataFrame([resume(s, a) for a, s in sorted(solaire.items())])
print(tableau.to_string(index=False))

# Valeurs de technologie par année (libellés stables ?)
for a, s in sorted(solaire.items()):
    if "technologie" in s:
        print(a, s["technologie"].fillna("NaN").value_counts().to_dict())

 annee  lignes_solaire_6dep  lignes_individuelles  puissance_MW_indiv  puissance_MW_agreg  remplissage_dateRacc  remplissage_injectee                                                           gestionnaires
  2017                 2537                  1438               354.0                21.9                   1.0                   NaN [Enedis, Energie développement services du Briançonnais, GreenAlp, RTE]
  2018                 2763                  1627               472.8                37.6                   1.0                   NaN [Enedis, Energie développement services du Briançonnais, GreenAlp, RTE]
  2019                 3040                  1847               608.7                46.6                   1.0                   NaN [Enedis, Energie développement services du Briançonnais, GreenAlp, RTE]
  2020                 3399                  2149               796.9               149.8                   1.0                 0.982 [Enedis, Energie développement services du

In [7]:
def puissance_indiv(s):
    s = s.copy()
    s["puis"] = pd.to_numeric(s["puisMaxInstallee"].str.replace(",", ".", regex=False), errors="coerce")
    if "technologie" in s:
        s = s[s["technologie"] == "Photovoltaïque"]
    if "nbInstallations" in s:
        s = s[pd.to_numeric(s["nbInstallations"], errors="coerce") == 1]
    elif "codeEICResourceObject" in s:
        s = s[s["codeEICResourceObject"].notna()]
    return s.groupby("codeDepartement")["puis"].sum() / 1000

snap = pd.DataFrame({a: puissance_indiv(s) for a, s in solaire.items() if a < 2026})
print(snap.round(1))                       # puissance vue par chaque fichier annuel
print((snap - annuel[list(snap.columns)]).round(1))   # écart avec ta reconstruction

                  2017   2018   2019   2020   2021   2022   2023   2024   2025
codeDepartement                                                               
04               205.2  207.2  231.2  265.8  354.7  365.8  382.1  431.6  486.9
05                34.5   52.8   62.1   77.6  103.5  134.9  149.5  209.3  229.3
07                16.5   30.8   52.5   78.7  106.4  115.9  129.0  139.5  153.6
26                31.2   61.5  100.5  139.6  194.2  234.6  257.3  303.2  367.1
38                13.4   28.7   35.2   61.7   79.2  111.4  167.4  220.6  306.5
84                50.6   91.7  127.2  173.4  190.0  211.9  233.9  278.1  316.8
                 2017  2018  2019  2020  2021  2022  2023  2024  2025
codeDepartement                                                      
04              -75.4 -84.5 -68.0 -62.7   3.2   6.6   3.0   2.0   9.5
05              -18.8 -18.1 -17.8  -6.7  -0.5   0.5  -0.6   0.9   1.3
07              -31.5 -30.3 -13.5  -0.3  -0.3  -5.7  -1.2   0.2   0.1
26              -5

In [8]:
def num(s, cols):
    s = s.copy()
    for c in cols:
        if c in s: s[c] = pd.to_numeric(s[c].str.replace(",", ".", regex=False), errors="coerce")
    return s

# TEST A : qualité de la puissance dans chaque fichier (lignes individuelles)
def qualite(s):
    i = num(s, ["puisMaxInstallee", "puisMaxRac", "nbInstallations"])
    i = i[i["nbInstallations"] == 1]
    return pd.Series({"lignes": len(i),
                      "puis_NaN": i["puisMaxInstallee"].isna().sum(),
                      "puis_<=0": (i["puisMaxInstallee"] <= 0).sum(),
                      "puis_mediane": i["puisMaxInstallee"].median(),
                      "puis_moyenne": i["puisMaxInstallee"].mean(),
                      "rac_somme_MW": i["puisMaxRac"].sum() / 1000})
print(pd.DataFrame({a: qualite(s) for a, s in solaire.items()}).T.round(1))

# TEST B : mêmes installations entre 2017 et 2026 ? (clé : code EIC)
a17 = num(solaire[2017], ["puisMaxInstallee", "nbInstallations"])
a26 = num(solaire[2026], ["puisMaxInstallee", "nbInstallations"])
i17 = a17[a17["nbInstallations"] == 1].drop_duplicates("codeEICResourceObject").set_index("codeEICResourceObject")
i26 = a26[a26["nbInstallations"] == 1].drop_duplicates("codeEICResourceObject").set_index("codeEICResourceObject")
communs = i17.index.intersection(i26.index)
print(len(i17), len(i26), len(communs))
cmp = pd.DataFrame({"p17": i17.loc[communs, "puisMaxInstallee"],
                    "p26": i26.loc[communs, "puisMaxInstallee"],
                    "commune": i26.loc[communs, "commune"]})
cmp["ecart"] = cmp["p26"] - cmp["p17"]
print((cmp["p17"] == cmp["p26"]).mean())            # part des installations à puissance identique
print(cmp["p17"].isna().sum(), (cmp["p17"] <= 0).sum())
print(cmp.sort_values("ecart", ascending=False).head(10))

# TEST C : production dans les fichiers 2017-2019
for a in [2017, 2018, 2019]:
    s = solaire[a]
    for c in ["energieAnnuelleInjectee", "energieAnnuelleProduite"]:
        print(a, c, round(s[c].notna().mean(), 3))

      lignes  puis_NaN  puis_<=0  puis_mediane  puis_moyenne  rac_somme_MW
2017  1438.0     389.0       0.0          95.0         337.5         593.7
2018  1627.0     316.0       0.0          98.8         360.6         713.2
2019  1847.0     204.0       0.0          99.0         370.5         789.4
2020  2149.0      87.0       0.0          99.0         386.5         898.4
2021  2604.0       3.0       0.0          99.0         395.3        1024.6
2022  3095.0       3.0       0.0          99.0         380.0        1167.6
2023  3672.0       3.0       0.0          99.0         359.5        1306.6
2024  4553.0       6.0       0.0         100.0         348.0        1542.6
2025  5682.0       0.0       0.0         100.0         327.4        1802.2
2026  6332.0       0.0       0.0         108.0         315.8        1945.5
1438 6332 1399
0.5396711937097927
366 0
                           p17       p26                      commune   ecart
codeEICResourceObject                                    

In [9]:
def to_num(x):
    return pd.to_numeric(x.str.replace(",", ".", regex=False), errors="coerce")

def to_date(x):
    d = pd.to_datetime(x, format="%d/%m/%Y", errors="coerce")
    manque = d.isna() & x.notna()
    if manque.any():
        d[manque] = pd.to_datetime(x[manque], errors="coerce")   # autre format éventuel
    return d

morceaux = []
for annee, s in sorted(solaire.items()):
    d = s[(to_num(s["nbInstallations"]) == 1) & (s["technologie"] == "Photovoltaïque")].copy()
    col_prod = ("energieAnnuelleGlissanteInjectee"
                if "energieAnnuelleGlissanteInjectee" in d else "energieAnnuelleInjectee")
    d["edition"] = annee
    d["puis_installee"] = to_num(d["puisMaxInstallee"])
    d["puis_rac"] = to_num(d["puisMaxRac"])
    d["injectee_kwh"] = to_num(d[col_prod])
    d["date_racc"] = to_date(d["dateRaccordement"])
    morceaux.append(d[["edition", "codeDepartement", "codeEICResourceObject", "commune",
                       "codeINSEECommune", "date_racc", "puis_installee", "puis_rac", "injectee_kwh"]])
panel = pd.concat(morceaux, ignore_index=True)

# Puissance en cascade, avec trace de la source
p26 = (panel[panel["edition"] == 2026].drop_duplicates("codeEICResourceObject")
       .set_index("codeEICResourceObject")["puis_installee"])
panel["p26"] = panel["codeEICResourceObject"].map(p26)
panel["puis_kw"] = panel["puis_installee"]
panel["source_puis"] = "installee"
for col, nom in [("puis_rac", "raccordement"), ("p26", "fichier_2026")]:
    m = panel["puis_kw"].isna() & panel[col].notna()
    panel.loc[m, "puis_kw"] = panel.loc[m, col]
    panel.loc[m, "source_puis"] = nom
panel.loc[panel["puis_kw"].isna(), "source_puis"] = "manquante"

# CONTRÔLE 1 : origine des puissances, par édition
print(panel.groupby(["edition", "source_puis"]).size().unstack(fill_value=0))

# CONTRÔLE 2 : total du panel vs reconstruction (annuel : table puissance cumulée, colonnes = années)
tot = panel.groupby("edition")["puis_kw"].sum().div(1000).round(1)
print(pd.DataFrame({"panel_MW": tot, "recon_MW": annuel.sum().reindex(tot.index).round(1)}))

# CONTRÔLE 3 : unité et période de la production (installations raccordées avant la période)
seuil = pd.to_datetime(panel["edition"].astype(str) + "-01-01")
seuil = seuil.where(panel["edition"] != 2026, pd.Timestamp("2025-09-01"))
mat = panel[panel["date_racc"] < seuil].copy()
mat["kwh_par_kw"] = mat["injectee_kwh"] / mat["puis_kw"]
print(mat.groupby("edition")["kwh_par_kw"].describe()[["count", "25%", "50%", "75%"]].round(0))

# CONTRÔLE 4 : date de la photographie de chaque édition
print(panel.groupby("edition")["date_racc"].max())

# Sauvegarde (après avoir regardé les contrôles ci-dessus)
# panel.to_csv("dataset/data clean/centrales_pv_6dep.csv", index=False, sep=";", encoding="utf-8")

source_puis  installee  raccordement
edition                             
2017              1048           370
2018              1311           312
2019              1643           203
2020              2062            86
2021              2600             3
2022              3090             3
2023              3669             3
2024              4547             6
2025              5680             0
2026              6322             0
         panel_MW  recon_MW
edition                    
2017        572.4     584.7
2018        700.4     706.2
2019        782.1     785.0
2020        893.4     894.9
2021       1032.9    1031.6
2022       1183.2    1184.6
2023       1327.8    1331.1
2024       1583.9    1581.1
2025       1860.2    1848.4
2026       1998.0    1998.0
          count     25%     50%     75%
edition                                
2017     1281.0  1238.0  1382.0  1513.0
2018     1419.0  1164.0  1289.0  1405.0
2019     1604.0  1229.0  1375.0  1509.0
2020     1830.0  119